In [1]:
import time
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel

from cic_pca4_runner import load_balanced_sample

In [2]:
qdf, numeric_feature_cols, dataset_summary = load_balanced_sample(
    samples_per_class=100,
    random_state=42,
)

Dataset schema: 2,121,949 rows, 334 shared columns, 314 numeric features


Scanning class labels...


Loading 100 benign and 100 malware rows...


In [3]:
print("Combined shared-column shape:", (
    dataset_summary["combined_rows"],
    dataset_summary["shared_column_count"],
))
print("Numeric feature count:", len(numeric_feature_cols))
print("\nSample architecture counts:")
print(qdf["Arch"].value_counts())
print("\nFull-dataset malware family counts:")
print(pd.Series(dataset_summary["family_counts"]))

Combined shared-column shape: (2121949, 334)
Numeric feature count: 314

Sample architecture counts:
Arch
arm       65
mipsel    55
x86       55
mips      25
Name: count, dtype: int64

Full-dataset malware family counts:
Agent            216
Benign       1046006
DarkNexus      23365
Gafgyt          4937
Generic         5416
Mirai         777418
Rudedevil         85
Tsunami          359
Unknown       264147
dtype: int64


In [4]:
N_QFEATURES = 4

feature_map = ZZFeatureMap(
    feature_dimension=N_QFEATURES,
    reps=2,
    entanglement="linear"
)

kernel = FidelityQuantumKernel(feature_map=feature_map)

model = SVC(kernel="precomputed")

/tmp/ipykernel_24491/4221885500.py:3: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


In [5]:
target_col = "MalwareFamily"

family_to_test = "Benign"

print("Testing malware family:", family_to_test)

Testing malware family: Benign


In [6]:
qX = qdf[numeric_feature_cols].replace([np.inf, -np.inf], np.nan)

qy = np.where(qdf[target_col].astype(str) == str(family_to_test), 0, 1)

print("qX shape:", qX.shape)
print("qy counts:")
print(pd.Series(qy).value_counts())

qX shape: (200, 314)
qy counts:
0    100
1    100
Name: count, dtype: int64


In [7]:
qX_train, qX_test, qy_train, qy_test = train_test_split(
    qX,
    qy,
    test_size=0.3,
    random_state=42,
    stratify=qy
)

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

# Convert to numpy arrays
qX_train = np.asarray(qX_train, dtype=float)
qX_test = np.asarray(qX_test, dtype=float)

# Replace inf values with nan so the imputer can handle them
qX_train = np.where(np.isfinite(qX_train), qX_train, np.nan)
qX_test = np.where(np.isfinite(qX_test), qX_test, np.nan)

# Fill missing values
imputer = SimpleImputer(strategy="median", keep_empty_features=True)
qX_train = imputer.fit_transform(qX_train)
qX_test = imputer.transform(qX_test)

# Standardize
scaler = StandardScaler()
qX_train = scaler.fit_transform(qX_train)
qX_test = scaler.transform(qX_test)

# Reduce 314 shared numeric features down to 4 quantum features
pca = PCA(n_components=N_QFEATURES, random_state=42)
qX_train = pca.fit_transform(qX_train)
qX_test = pca.transform(qX_test)

# Scale to range used by the quantum feature map
range_scaler = MinMaxScaler(feature_range=(-1, 1))
qX_train = range_scaler.fit_transform(qX_train)
qX_test = range_scaler.transform(qX_test)

# Final check
print("qX_train shape:", qX_train.shape)
print("qX_test shape:", qX_test.shape)
print("Any NaN train:", np.isnan(qX_train).any())
print("Any NaN test:", np.isnan(qX_test).any())
print("N_QFEATURES:", N_QFEATURES)

qX_train shape: (140, 4)
qX_test shape: (60, 4)
Any NaN train: False
Any NaN test: False
N_QFEATURES: 4


In [9]:
start = time.process_time()

K_train = kernel.evaluate(qX_train, qX_train)
K_train = np.asarray(K_train, dtype=float)
K_train = np.nan_to_num(K_train, nan=0.0, posinf=1.0, neginf=0.0)

model.fit(K_train, qy_train);

In [10]:
# Reuse the training-fitted preprocessing above. Do not refit it here.

In [11]:
K_test = kernel.evaluate(qX_test, qX_train)
K_test = np.asarray(K_test, dtype=float)
K_test = np.nan_to_num(K_test, nan=0.0, posinf=1.0, neginf=0.0)

y_pred = model.predict(K_test)
decision_scores = model.decision_function(K_test)

end = time.process_time()

In [12]:
accuracy = accuracy_score(qy_test, y_pred)
balanced_accuracy = balanced_accuracy_score(qy_test, y_pred)
roc_auc = roc_auc_score(qy_test, decision_scores)

print(f"Accuracy: {accuracy}")
print(f"Balanced accuracy: {balanced_accuracy}")
print(f"ROC AUC: {roc_auc}")
print(f"CPU time: {end - start:.6f} seconds")

print("\nClassification Report:")
print(classification_report(qy_test, y_pred, target_names=["Benign", "Malware"]))

print("\nConfusion Matrix:")
print(confusion_matrix(qy_test, y_pred))

Accuracy: 0.5666666666666667
Balanced accuracy: 0.5666666666666667
ROC AUC: 0.6633333333333333
CPU time: 104.650026 seconds

Classification Report:
              precision    recall  f1-score   support

      Benign       0.56      0.63      0.59        30
     Malware       0.58      0.50      0.54        30

    accuracy                           0.57        60
   macro avg       0.57      0.57      0.56        60
weighted avg       0.57      0.57      0.56        60


Confusion Matrix:
[[19 11]
 [15 15]]
